# 🛒 Notebook 05 — Cập nhật Shopee Recommender Model

**Mục tiêu:**
- Gộp dữ liệu từ Shopee (`data.xlsx` & `shopee_product_desc.json`) để xây dựng một tập catalog mới với hơn 15.000 sản phẩm.
- Huấn luyện lại mô hình Recommender (Content-based Filtering) sử dụng cấu trúc TF-IDF và Cosine Similarity.
- Xuất mô hình mới để tích hợp vào API backend, giúp chatbot AI tư vấn sâu hơn dựa trên `description`.

**Đầu vào:** `datas/recommend_dataset/data.xlsx`, `datas/recommend_dataset/shopee_product_desc.json`  
**Đầu ra:** `datas/models/shopee_catalog_new.csv`, `datas/models/shopee_recommender.pkl`


In [5]:
import pandas as pd
import re
import urllib.parse
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Khai báo đường dẫn
excel_path = "../datas/recommend_dataset/data.xlsx"
json_path = "../datas/recommend_dataset/shopee_product_desc.json"
out_csv = "../datas/models/shopee_catalog_new.csv"
out_model = "../datas/models/shopee_recommender.pkl"


## 1. Trích xuất & Gộp dữ liệu (Merge Data)

In [6]:
def extract_from_shopee_url(url):
    try:
        url_str = str(url)
        decoded_url = urllib.parse.unquote(url_str)
        # Lấy tên, shop_id, item_id
        match = re.search(r'shopee\.vn/([^/]+)-i\.(\d+)\.(\d+)', decoded_url)
        if match:
            return match.group(1).replace('-', ' '), match.group(2), match.group(3)
    except:
        pass
    return None, None, None

# Đọc JSON
df_json = pd.read_json(json_path)
extracted = df_json['url'].apply(extract_from_shopee_url)
df_json['name'] = [x[0] for x in extracted]
df_json['item_id'] = [x[1] for x in extracted]
df_json['shop_id'] = [x[2] for x in extracted]
df_json = df_json.rename(columns={'desc': 'description'}).dropna(subset=['item_id'])
df_json['product_id'] = 'shp_' + df_json['shop_id'].astype(str) + '_' + df_json['item_id'].astype(str)
df_json['location'] = ""

# Đọc Excel
df_excel = pd.read_excel(excel_path)
df_excel['product_id'] = 'shp_' + df_excel['shop_id'].astype(str) + '_' + df_excel['item_id'].astype(str)
df_excel = df_excel.rename(columns={'shop_location': 'location'})
df_excel['description'] = ""
df_excel['url'] = ""

# Gộp (Concat)
df_combined = pd.concat([
    df_json[['product_id', 'name', 'description', 'url', 'location']], 
    df_excel[['product_id', 'name', 'description', 'url', 'location']]
], ignore_index=True)

# Xử lý missing & duplicate
df_combined['has_desc'] = df_combined['description'].apply(lambda x: 1 if pd.notna(x) and x != "" else 0)
df_combined = df_combined.sort_values(by='has_desc', ascending=False).drop_duplicates(subset=['product_id'], keep='first')
df_combined['name'] = df_combined['name'].fillna("Sản phẩm Shopee")
df_combined['description'] = df_combined['description'].fillna("")
df_combined['location'] = df_combined['location'].fillna("Toàn quốc")
df_combined['category'] = "Shopee"

df_combined['combined_text'] = (df_combined['name'] + " " + df_combined['description']).str.strip()

print(f"Tổng số sản phẩm sau khi gộp: {len(df_combined)}")
df_combined.to_csv(out_csv, index=False, encoding='utf-8-sig')
df_combined.head(3)


Tổng số sản phẩm sau khi gộp: 15463


,product_id,name,description,url,location,has_desc,category,combined_text
13113,shp_18041140032_552312266,Đồng Hồ Nữ Chính Hãng ULZZANG U11 TD6 Dây Kim ...,THÔNG TIN VỀ SẢN PHẨM\n⏩ Chất liệu dây đeo : D...,https://shopee.vn/Đồng-Hồ-Nữ-Chính-Hãng-ULZZAN...,,1,Shopee,Đồng Hồ Nữ Chính Hãng ULZZANG U11 TD6 Dây Kim ...
13112,shp_18951846491_107745411,Đồng hồ nữ dây da mặt tròn Gedi siêu xinh cho ...,🌹 Đồng hồ nữ dây da mặt tròn Gedi\n===========...,https://shopee.vn/Đồng-hồ-nữ-dây-da-mặt-tròn-G...,,1,Shopee,Đồng hồ nữ dây da mặt tròn Gedi siêu xinh cho ...
13111,shp_3441162625_134786926,Đồng hồ nam dây da chính hãng Casio MTP V001L ...,Các tính năng:\n\n• Chống nước 3ATM\n\n• Thiết...,https://shopee.vn/Đồng-hồ-nam-dây-da-chính-hãn...,,1,Shopee,Đồng hồ nam dây da chính hãng Casio MTP V001L ...


## 2. Huấn luyện mô hình Recommender (TF-IDF)

In [7]:
import joblib
import sys
sys.path.append('..')
from src.models.recommender import ProductRecommender

print("Khởi tạo mô hình ProductRecommender...")
recommender = ProductRecommender(ngram_range=(1, 2), max_features=40000)

print("Đang huấn luyện (fit) mô hình trên text kết hợp (Tên + Mô tả)...")
recommender.fit(df_combined, text_column='combined_text')

# Lưu mô hình
joblib.dump(recommender, out_model)
print(f"Mô hình đã được lưu tại: {out_model}")


Khởi tạo mô hình ProductRecommender...
Đang huấn luyện (fit) mô hình trên text kết hợp (Tên + Mô tả)...
Mô hình đã được lưu tại: ../datas/models/shopee_recommender.pkl


## 3. Chạy thử Gợi ý (Test Recommendation)

In [8]:
query = "bánh tráng trộn siêu cay"
print(f"Kết quả gợi ý cho query: '{query}'\n" + "-"*50)

results = recommender.recommend_by_query(query, top_k=5)
results[['product_id', 'name', 'location', 'similarity_score']]


Kết quả gợi ý cho query: 'bánh tráng trộn siêu cay'
--------------------------------------------------


,product_id,name,location,similarity_score
82,shp_4938329028_272333943,Bánh tráng hồng hạnh sate muối tắc,,0.3611
3347,shp_4959124161_33410512,BÁNH TRÁNG SATE TẮC MUỐI,,0.3123
124,shp_10079004677_171780566,Bánh tráng trộn thập cẩm nhiều topping bánh tr...,,0.3081
76,shp_4150870058_299404199,bánh tráng muối tỏi xike theo kg (1kg 65k),,0.3021
32,shp_13766270066_152098464,Bánh tráng phơi sương muối nhuyễn sate ớt tắc ...,,0.2990
